# Session 25: LSTM — Architecture & Gates

### Task 1: Keras LSTM for Step Counts
A simple Keras script to build an LSTM layer that predicts the next value in a sequence of daily step counts.

In [ ]:
pip install torch

In [4]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

# 1. Prepare dummy data: Predict next day based on previous 3 days
# Input shape: (samples, time_steps, features)
X = np.array([
    [[4000], [4200], [4300]], 
    [[4200], [4300], [4100]]
])
# Targets (Next day's steps)
y = np.array([4100, 4500])

# 2. Build the LSTM Model
model = Sequential()
# LSTM layer with 50 units. input_shape = (3 time steps, 1 feature)
model.add(LSTM(50, activation='relu', input_shape=(3, 1)))
# Output layer predicting a single numerical value
model.add(Dense(1))

# 3. Compile the model
model.compile(optimizer='adam', loss='mse')

# 4. Train
model.fit(X, y, epochs=100, verbose=0)
print("Model trained successfully!")

c:\Users\param\machine learning\deep learning\CNN project\cnn\.venv\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model trained successfully!


### Task 2: Demonstrating Vanishing Gradients in a Basic RNN
Using PyTorch to manually pass a sequence through a basic RNN layer and check the gradients of the hidden states. You will notice how the gradient at the earliest step (step 0) is exponentially smaller than at the final step!

In [6]:
import torch
import torch.nn as nn

# Setup a basic RNN
rnn = nn.RNN(input_size=1, hidden_size=1, batch_first=True)

# Create a long sequence of 50 time steps
seq_length = 50
x = torch.ones(1, seq_length, 1, requires_grad=True) # Input sequence
h0 = torch.zeros(1, 1, 1) # Initial hidden state

# Forward pass
output, hn = rnn(x, h0)

# Simulate a loss based ONLY on the very last output
loss = output[0, -1, 0] ** 2
loss.backward()

# Check the gradients of the input x at different time steps
print(f"Gradient at step 49 (latest): {x.grad[0, 49, 0].item():.6f}")
print(f"Gradient at step 25 (middle): {x.grad[0, 25, 0].item():.6f}")
print(f"Gradient at step 0 (earliest): {x.grad[0, 0, 0].item():.6f}")

Gradient at step 49 (latest): -0.009593
Gradient at step 25 (middle): -0.000029
Gradient at step 0 (earliest): 0.000000


### Task 3: LSTM Cell Architecture & Gates


**1-Line Explanations for Each Component:**
* **Cell State ($C_t$):** The core "highway" memory that carries long-term information straight through the entire sequence with minimal changes.
* **Forget Gate ($f_t$):** Looks at the new input and previous hidden state to decide which past memories in the cell state are no longer relevant and should be erased (multiplied by 0).
* **Input Gate ($i_t$):** Decides which specific parts of the new input data are important enough to be added to the long-term cell state.
* **Output Gate ($o_t$):** Filters the newly updated cell state to determine what the actual output (the new hidden state) should be for the current time step.

### Task 4: Manual Walkthrough (Zomato Orders)
Let's trace the sequence `['pizza', 'burger', 'pasta', 'pizza']` for the first two steps using binary values (1 = Keep/Write, 0 = Discard/Ignore).

**Time Step 1: Input = 'pizza'**
* **Forget Gate:** Previous state is empty, so it outputs **0** (nothing to forget).
* **Input Gate:** 'pizza' is the user's first distinct preference. Output is **1** (Write 'pizza' context to the cell state).
* **Output Gate:** Outputs **1**. The hidden state now passes along the context: *"User likes fast food / Italian."*

**Time Step 2: Input = 'burger'**
* **Forget Gate:** Does the user still like fast food? Yes. Output is **1** (Keep the 'pizza' context in the long-term memory).
* **Input Gate:** 'burger' adds new distinct info (American fast food). Output is **1** (Add 'burger' to the cell state).
* **Output Gate:** Outputs **1**. The hidden state combines the cell state to pass along: *"User strongly prefers fast food items right now."*

### Task 5: Unrolling Through Time (Music App)

**What it means:**
"Unrolling an LSTM through time" means visualizing the single LSTM cell as a chain of multiple repeating cells, one for each step in a sequence. For a music app predicting your next song, unrolling means drawing the network so that Step 1 processes Song A, passes its hidden state to Step 2 which processes Song B, which passes its state to Step 3 for Song C. It is the exact same neural network layer, just stretched out chronologically to show how the context of Song A flows all the way to Song C to predict Song D.